In [3]:
# exclamation means that it is a shell command
!echo 1

1


In [ ]:
import re
import sys
import subprocess

def naive_parse(answer):
    out = []
    start = False
    end = False
    for l in reversed(list(answer)):
        if l in '0123456789' and not end:
            start = True
            out.append(l)
        else:
            if start:
                end = True
        
    out = reversed(out)
    return ''.join(out)

def extract_answer(completion):
    try:
        result = re.findall(r'\\boxed\{(\d+\.?\d*)\}', completion)
        if not len(result):
            result =naive_parse(completion)
            result = int(float(result))
        else:
            result = result[-1]
            result = int(float(result))
        if result < 0:
            result = "Invalid"
    except Exception as e:
        print(e)
        result = "Invalid"
    return result

def return_last_print(output, n):
    lines = output.strip().split('\n')
    if lines:
        return lines[n]
    else:
        return ""

def process_code(code, return_shell_output=False):
    
    def repl(match):
        if "real" not in match.group():
            return "{}{}".format(match.group()[:-1], ', real=True)')
        else:
            return "{}{}".format(match.group()[:-1], ')')
    code = re.sub(r"symbols\([^)]+\)", repl, code)

    if return_shell_output:
        code = code.replace('\n', '\n    ')
            # Add a try...except block
        code = "\ntry:\n    from sympy import *\n{}\nexcept Exception as e:\n    print(e)\n    print('FAIL')\n".format(code)
    
    if not return_shell_output:
        print(code)
    with open('temp.py', 'w') as fout:
        fout.write(code)
    
    batcmd = 'timeout 7 ' + sys.executable + ' temp.py'
    try:
        shell_output = subprocess.check_output(batcmd, shell=True).decode('utf8')
        return_value = return_last_print(shell_output, -1)
        if return_shell_output:
            if return_value=='FAIL':
                CODE_STATUS = False
                return_value = return_last_print(shell_output, -2)
                if "not defined" in return_value:
                    return_value+='\nTry checking the formatting and imports'
            else:
                CODE_STATUS = True
            return return_value, CODE_STATUS  
        code_output = round(float(return_value)) % 1000
    except Exception as e:
        code_output = -1
    
    if return_shell_output:
        if code_output==-1:
            CODE_STATUS = False
        else:
            CODE_STATUS = True
        return code_output, CODE_STATUS  
    
    
    return code_output

In [ ]:
from vllm import LLM, SamplingParams
llm = LLM(model=MODEL_PATH,
    dtype="half",
    enforce_eager=True,
    swap_space=4,
    gpu_memory_utilization=0.99,
    disable_custom_all_reduce=False,
    kv_cache_dtype="fp8_e5m2",
    max_model_len=2800,
    tensor_parallel_size=1)

In [ ]:
from collections import Counter
import torch
import gc
tool_instruction = '\nPlease integrate natural language reasoning with programs to solve the problem above, and put your final answer within \\boxed{}.'

def vllm_inference(problem, num_sequences, T):
    sampling_params = SamplingParams(temperature=T, top_p=1.0, n=num_sequences, max_tokens=2048)
    prompt = problem + tool_instruction
    generated_texts = llm.generate(prompt, sampling_params)
    results = []
    result_texts = []
    for generated_text in generated_texts[0].outputs:
        text = generated_text.text
        try:
            code_text = text.split('```python')[-1].split("```")[0]
            code_result, CODE_STATUS = process_code(code_text, return_shell_output=True)
            result = int(float(code_result.strip()))
            if result >= 0:
                results.append(result % 1000)
                result_texts.append((prompt + text).split("```output")[0])
        except Exception as e:
            pass
    return results, result_texts


def vllm_round_inference(problem, num_sequences, T, rounds):
    prompt = problem + tool_instruction
    texts = [prompt for _ in range(num_sequences)]
    results = []
    result_texts = []
    for i in range(rounds):
        sampling_params = SamplingParams(temperature=T, top_p=1.0, n=1, max_tokens=1024, stop="```output")
        generated_texts = llm.generate(texts, sampling_params)
        next_round_texts = []
        completed = []
        print(len(generated_texts))
        for j, (text, generated_text) in enumerate(zip(texts, generated_texts)):
            new_text = generated_text.outputs[0].text
            completed = False
            if '```python' in new_text:
                try:
                    code_text = new_text.split('```python')[-1].split("```")[0]
                    code_result, CODE_STATUS = process_code(code_text, return_shell_output=True)
                    try:
                        result = int(float(code_result.strip()))
                        if result >= 0:
                            results.append(result % 1000)
                            result_texts.append((prompt + "```python" + new_text.split('```python')[-1]).split("```output")[0])
                        completed = True
                    except Exception as e:
                       code_result += "The code output is not an integer, final answer should be an integer."
                except Exception as e:
                    code_result = "Code execution fail"
                    pass
                if completed == False:
                    next_round_texts.append(text + new_text + "```output\n" + code_result + "\n```")

        texts = next_round_texts
        print("remaining text:", len(texts))
        if texts == []:
            break
    return results, result_texts

            


In [ ]:
def orm_score(reward_model, tokenizer, text):
    input_id = torch.tensor([tokenizer.encode(text)]).to(reward_model.device)
    with torch.no_grad():
        logits = reward_model(input_id).logits
        score = logits[0].mean(dim=-1).sigmoid().cpu().tolist()[-1]
    return score

def weighted_majority_voting(answers, scores):
    max_weight_sum = -1
    number_weight_dict = {}
    max_weight_answer = None
    for answer, score in zip(answers, scores):
        if answer not in number_weight_dict:
            number_weight_dict[answer] = score
        else:
            number_weight_dict[answer] += score
        if number_weight_dict[answer] > max_weight_sum:
            max_weight_sum = number_weight_dict[answer]
            max_weight_answer = answer
    return max_weight_answer

def weighted_geometric_mean_times_num(answers, scores):
    import math
    max_weight = -1
    number_weight_dict = {}
    max_weight_answer = None
    for answer, score in zip(answers, scores):
        if answer == 0:
            score /= 10
        if answer not in number_weight_dict:
            number_weight_dict[answer] = [score]
        else:
            number_weight_dict[answer].append(score)
    for answer in number_weight_dict.keys():
        score_list = number_weight_dict[answer]
        weight = len(score_list) * math.prod(score_list) ** (1 / len(score_list))
        if weight > max_weight:
            max_weight = weight
            max_weight_answer = answer
    return max_weight_answer





In [ ]:
from collections import Counter
import os


temperature = 0.9
n_repetitions = 1
batch_size = 42
device = torch.device("cuda:1")
if Test == True:
    question_start_times = []
    tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/deepseek-finetune/transformers/orm_fp32/1/orm_fp32")
    reward_model = AutoModelForCausalLM.from_pretrained("/kaggle/input/deepseek-finetune/transformers/orm_fp32/1/orm_fp32",
                                                       torch_dtype="auto",
                                                        trust_remote_code=True)
    reward_model.to(device)
    reward_model.eval()
    for i, (test, sample_sumbmission) in enumerate(iter_test):
        TIME = time.time()
        if TIME - START_TIME >= 31800:
            break
        candidate_answers = []
        scores = []
        final_answer = "None"


        question_start_times.append(time.time())
        problem = test['problem'].values[0]
        for jj in range(n_repetitions):
            batch_results, batch_results_texts = vllm_round_inference(problem, batch_size, temperature, 2)
            candidate_answers.extend(batch_results)

        pairs = []
        if candidate_answers != []:
            for t, text in enumerate(batch_results_texts):
                scores.append(orm_score(reward_model, tokenizer, text))
                pairs.append((candidate_answers[t], scores[t]))
            print(pairs)
            final_answer = weighted_geometric_mean_times_num(candidate_answers, scores)
            sample_sumbmission['answer'] = final_answer
        
        else:
            sample_sumbmission['answer'] = 0
        
        env.predict(sample_sumbmission)






In [9]:
import torch

a = torch.tensor([[1, 2, 3]]) # 1x3
b = torch.tensor([[4, 5]]) # 1x2
b = b.unsqueeze(-1) # 1x2x1
b1 = (b == 4).float()
a, b, b1

(tensor([[1, 2, 3]]),
 tensor([[[4],
          [5]]]),
 tensor([[[1.],
          [0.]]]))

In [16]:
a * b1, a.unsqueeze(1) * b1

(tensor([[[1., 2., 3.],
          [0., 0., 0.]]]),
 tensor([[[1., 2., 3.],
          [0., 0., 0.]]]))

In [15]:
(b1 * a.unsqueeze(1)).shape, b1.shape, a.unsqueeze(1).shape

(torch.Size([1, 2, 3]), torch.Size([1, 2, 1]), torch.Size([1, 1, 3]))

In [19]:
import numpy as np

x = np.float16(1.0)
y = np.nextafter(x, 2.0)
x, y, y - x


(1.0, 1.0000000000000002, 2.220446049250313e-16)